# X-ray 이물 검출 YOLOv3 (리팩토링)
원본 `yolov3_20201200.ipynb`와 동일한 로직(라이브러리 준비 → 데이터 확인 → 재정리 → 라벨링 → 분리 → 학습 → 평가)을 경로 하드코딩(`C:/test1/...`) 없이 재구성한 노트북입니다.

- 실행 커널: conda 환경 **KAMP**
- 모든 경로는 `CONFIG` 한 곳에서 관리합니다.

# [단계 ①] 설정 및 라이브러리 불러오기

In [ ]:
import os, random, shutil, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image
import torch

# ---- 경로/실행 설정 (필요 시 여기만 수정) ----
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "yolov3").exists())   # KAMP 프로젝트 루트
YOLO_DIR = ROOT / "src" / "yolov3"              # train.py / detect.py / test.py 위치
LABEL_TOOL = None   # OpenLabeling 은 저장소에서 제거됨 (https://github.com/Cartucho/OpenLabeling)
LEGACY = ROOT / "4. X-ray 검사장비 AI 데이터셋" / "dataset" / "test1" / "yolov3"   # 원본 실습 폴더 (원본 이미지 출처)
RAW_DIR  = LEGACY / "X선이물검출기(06.23_09.22)"        # 원본 (호기별/SN..._NgImage)
SORTED_DIR = LEGACY / "X선이물검출기"                    # 호기별 재정리 결과
DATA_DIR = ROOT / "data" / "samples"                    # 실습 세트 (images/, labels/)
IMAGE_DIR  = DATA_DIR / "images"
RUN_DIR  = ROOT / "runs" / "baseline" / "_legacy_notebook"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CFG = "yolov3-spp.cfg"
DATA = (DATA_DIR / "custom.data").as_posix()           # 절대경로 (스크립트는 yolov3 폴더에서 실행됨)
NAMES = (ROOT / "data" / "classes.names").as_posix()
WEIGHTS = (ROOT / "weights" / "legacy" / "last_806ep_plus2.pt").as_posix()
os.environ["PYTHONUTF8"] = "1"                            # 한글 경로 파일을 하위 프로세스가 UTF-8로 읽도록
DEVICE = "" if torch.cuda.is_available() else "cpu"     # GPU 없으면 자동으로 cpu
print("ROOT:", ROOT)
print("CUDA:", torch.cuda.is_available(), "| device arg:", repr(DEVICE))
assert YOLO_DIR.exists(), f"yolov3 폴더를 찾을 수 없습니다: {YOLO_DIR}"

In [ ]:
def run_script(script, *args):
    """yolov3 폴더를 cwd로 하여 현재 커널(KAMP)의 파이썬으로 스크립트를 실행한다."""
    cmd = [sys.executable, "-u", script, *map(str, args)]
    print(">", " ".join(cmd))
    subprocess.run(cmd, cwd=YOLO_DIR, check=True)

def device_args():
    return ["--device", DEVICE] if DEVICE else []

def show_image(path, title=None):
    plt.imshow(Image.open(path)); plt.title(title or Path(path).name); plt.axis("off"); plt.show()

# [단계 ②] 데이터 종류 및 개수 확인

In [ ]:
from collections import Counter
if RAW_DIR.exists():
    files = [p for p in RAW_DIR.rglob("*") if p.is_file()]
    print("원본 파일 수:", len(files), dict(Counter(p.suffix.lower() for p in files)))
    show_image(next(p for p in files if p.suffix.lower() == ".bmp"))
else:
    print("원본 폴더 없음(생략):", RAW_DIR)

## 이미지 재정리 (원본 → 호기별 NgImage만 복사)

In [ ]:
def reorganize_raw(raw_dir=RAW_DIR, out_dir=SORTED_DIR):
    """raw_dir/<N호기(...)>/<...NgImage> 안의 파일을 out_dir/<N>호기/ 로 복사한다."""
    if not raw_dir.exists():
        print("원본 폴더 없음:", raw_dir); return
    for unit in sorted(p for p in raw_dir.iterdir() if p.is_dir()):
        dst = out_dir / f"{unit.name[0]}호기"
        dst.mkdir(parents=True, exist_ok=True)
        for sub in unit.iterdir():
            if sub.is_dir() and sub.name.endswith("NgImage"):     # 원본: b[17:] == 'NgImage'
                for f in sub.iterdir():
                    shutil.copy2(f, dst / f.name)
    print("완료:", out_dir)

reorganize_raw()

# [단계 ③] 데이터 정제 (라벨링)
`OpenLabeling`으로 이미지를 라벨링하면 YOLO 좌표 txt가 `OpenLabeling-master/main/output/YOLO_darknet`에 저장됩니다.
이미지는 `yolov3/images`, txt는 `yolov3/labels`로 옮깁니다.

키: **D** 다음 / **A** 이전 / **S** 클래스 변경 / **Q** 종료 / 마우스 우클릭 라벨 삭제

In [ ]:
# 라벨링 도구 실행 (GUI 창이 열립니다. 필요할 때만 실행)
LAUNCH_LABEL_TOOL = False
if LAUNCH_LABEL_TOOL:
    subprocess.run([sys.executable, str(LABEL_TOOL)], cwd=LABEL_TOOL.parent, check=True)

In [ ]:
def collect_labeling_output(move=False):
    """OpenLabeling 결과(YOLO_darknet txt)를 yolov3/labels 로 옮기거나 복사한다."""
    src = LABEL_TOOL.parent / "output" / "YOLO_darknet"
    dst = DATA_DIR / "labels"; dst.mkdir(exist_ok=True)
    op = shutil.move if move else shutil.copy2
    n = 0
    for f in src.glob("*.txt") if src.exists() else []:
        op(str(f), dst / f.name); n += 1
    print(f"{n}개 라벨 {'이동' if move else '복사'} → {dst}")

# collect_labeling_output()

# [단계 ④] 데이터 특성 파악

In [ ]:
imgs   = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".bmp"})
labels = {p.stem for p in (DATA_DIR / "labels").glob("*.txt")}
print("images:", len(imgs), "| labels:", len(labels), "| 라벨 없는 이미지:", sum(p.stem not in labels for p in imgs))
if imgs: show_image(imgs[0])

# [단계 ⑤] 학습/평가 데이터 분리

In [ ]:
def split_data_set(image_dir=IMAGE_DIR, out_dir=DATA_DIR, test_ratio=0.2, seed=None):
    """jpg 이미지를 train/test 로 무작위 분할해 out_dir 의 train.txt / test.txt 에 기록한다.
    경로는 절대경로로 저장한다."""
    rng = random.Random(seed)
    files = sorted(f for f in Path(image_dir).iterdir() if f.suffix.lower() in {".jpg", ".jpeg"})
    test_set = set(rng.sample(range(len(files)), k=int(test_ratio * len(files))))
    rel = Path(image_dir).resolve().as_posix()   # 절대경로 (스크립트는 yolov3 폴더에서 실행됨)
    with open(out_dir / "train.txt", "w", encoding="utf-8") as ftr, open(out_dir / "test.txt", "w", encoding="utf-8") as fte:
        for i, f in enumerate(files):
            (fte if i in test_set else ftr).write(f"{rel}/{f.name}\n")
    print(f"train {len(files) - len(test_set)} / test {len(test_set)}")

split_data_set(seed=42)

In [ ]:
def write_custom_data(path=Path(DATA)):
    """custom.data 를 현재 위치 기준 절대경로로 생성한다(원본은 C:/test1/... 하드코딩)."""
    n_cls = len([l for l in Path(NAMES).read_text(encoding="utf-8").splitlines() if l.strip()])
    path.write_text(
        f"classes= {n_cls}\n"
        f"train={(DATA_DIR / 'train.txt').as_posix()}\n"
        f"valid={(DATA_DIR / 'test.txt').as_posix()}\n"
        f"names={NAMES}\n", encoding="utf-8")
    print(path.read_text(encoding="utf-8"))

write_custom_data()

## (추가) n400 묶음 단위 8:1:1 분할
위 단계의 12/3 분할은 실습용입니다. 평가용으로는 `scripts/make_split.py`가 `data/subsets/n400`을 촬영 묶음 단위로 train/val/test = 8:1:1로 나눕니다(결과: `data/splits/`).

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts" / "make_split.py")], check=True)
# 학습: --data <KAMP>/data/splits/val.data / 최종 평가: test.py --data <KAMP>/data/splits/test.data

# [단계 ⑥] 모델 구축 (의존성 설치)

In [ ]:
# KAMP 환경에는 이미 설치되어 있습니다. 새 환경에서만 주석을 풀어 실행하세요.
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=YOLO_DIR, check=True)

# [단계 ⑦] 모델 훈련
`last.pt` 가 마지막 epoch 가중치만 저장합니다(`--nosave`). GPU가 없으면 `DEVICE="cpu"`로 자동 전환됩니다.

In [ ]:
EPOCHS, BATCH = 15, 3     # 원본: GPU 150/5, CPU 15/3
run_script("train.py", "--epochs", EPOCHS, "--batch-size", BATCH, "--weights", WEIGHTS,
           "--cfg", CFG, "--data", DATA, "--nosave", *device_args())

# [단계 ⑧] 결과 분석 및 해석

In [ ]:
# 추론: 결과 이미지는 yolov3/result 에 저장
run_script("detect.py", "--weights", WEIGHTS, "--source", IMAGE_DIR.as_posix(), "--cfg", CFG,
           "--names", NAMES, "--output", (RUN_DIR / "result").as_posix(), *device_args())

In [ ]:
# 추론 결과 미리보기
for p in sorted((RUN_DIR / "result").glob("*.jp*g"))[:4]:
    show_image(p)

In [ ]:
# 평가 (mAP 등)
run_script("test.py", "--cfg", CFG, "--batch-size", BATCH, "--data", DATA, "--weights", WEIGHTS, *device_args())